[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hoax12/fable-pytorch/blob/main/fable-pytorch/notebooks/13_art_of_speed.ipynb)

# ⚒️ Act III · Quest 13 — The Art of Speed

*Measure first, then make it fast: profiling, compile, precision, quantization, and shipping.*

⬅️ [12_art_of_action](12_art_of_action.ipynb)  •  [14_final_boss](14_final_boss.ipynb) ➡️

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math, random

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {device}')
np.random.seed(0); random.seed(0)

# ══════════════════ ⚒️ FORGE GRADER — run me once ══════════════════
# Powers check() and xp_report(). Exercises give instant feedback + XP.
_XP = {"earned": 0, "done": set(), "checks": {}}

def _register(name, points, test, hint):
    _XP["checks"][name] = (points, test, hint)

def check(name, *answer):
    """Grade an exercise: check("ex1", your_answer). Repeatable until correct."""
    if name not in _XP["checks"]:
        print(f"unknown exercise: {name!r} — available: {list(_XP['checks'])}")
        return
    points, test, hint = _XP["checks"][name]
    try:
        ok = bool(test(*answer))
        err = None
    except Exception as e:
        ok, err = False, f"{type(e).__name__}: {e}"
    if ok:
        first = name not in _XP["done"]
        if first:
            _XP["done"].add(name)
            _XP["earned"] += points
        total = sum(p for p, _, _ in _XP["checks"].values())
        tag = f"+{points} XP" if first else "already earned"
        print(f"✅ {name} — correct! {tag}   ⚡ {_XP['earned']}/{total} XP")
    else:
        msg = f"❌ {name} — not yet."
        if err:
            msg += f" (your answer raised {err})"
        print(msg + f"\n   💡 hint: {hint}")

def xp_report():
    total = sum(p for p, _, _ in _XP["checks"].values())
    n = len(_XP["checks"])
    bar = "█" * int(10 * _XP["earned"] / max(total, 1)) or "░"
    print(f"⚡ XP: {_XP['earned']}/{total}   [{bar:<10}]   exercises: {len(_XP['done'])}/{n}")
    for name in _XP["checks"]:
        print(("  ✅ " if name in _XP["done"] else "  ⬜ ") + name)

## The optimizer's oath: *measure first*

Guessing at bottlenecks wastes weeks. Build a timing habit before touching any knob.

In [ ]:
import time

def stopwatch(fn, warmup=3, reps=20):
    for _ in range(warmup):
        fn()
    if device.type == "cuda":
        torch.cuda.synchronize()          # GPU work is async — always sync before timing!
    t0 = time.time()
    for _ in range(reps):
        fn()
    if device.type == "cuda":
        torch.cuda.synchronize()
    return (time.time() - t0) / reps * 1000

net = nn.Sequential(nn.Linear(512, 1024), nn.ReLU(), nn.Linear(1024, 1024), nn.ReLU(), nn.Linear(1024, 256)).to(device)
x = torch.randn(256, 512, device=device)
base_ms = stopwatch(lambda: net(x))
print(f"baseline forward: {base_ms:.2f} ms")

### Weapon 1 — batching (the free lunch)

The single biggest inference speedup most people never take: process many inputs per call.

In [ ]:
one = torch.randn(1, 512, device=device)
t_single = stopwatch(lambda: net(one))
t_batched = stopwatch(lambda: net(x))       # 256 at once
print(f"256 items one-by-one: ~{t_single * 256:7.1f} ms")
print(f"256 items in a batch:  {t_batched:7.2f} ms   ({t_single * 256 / t_batched:.0f}x faster)")

### Weapon 2 — `torch.compile` (PyTorch 2.x)

A JIT that fuses ops and generates optimized kernels. One line. Gains are largest on GPU;
on Windows/CPU it may fall back gracefully — which is why we guard it.

In [ ]:
try:
    fast = torch.compile(net)
    fast(x)                                  # first call compiles (slow), later calls are fast
    comp_ms = stopwatch(lambda: fast(x))
    print(f"eager: {base_ms:.2f} ms  |  compiled: {comp_ms:.2f} ms")
except Exception as e:
    print(f"torch.compile unavailable here ({type(e).__name__}) — try it on Colab GPU, where it shines.")

### Weapon 3 — mixed precision (GPU only)

float16/bfloat16 halves memory and can double throughput on tensor cores. The recipe (for Colab):

```python
scaler = torch.cuda.amp.GradScaler()
with torch.autocast(device_type="cuda", dtype=torch.float16):
    loss = model(x).sum()
scaler.scale(loss).backward()
scaler.step(opt); scaler.update()
```

In [ ]:
if device.type == "cuda":
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        y_half = net(x)
    print("autocast ran; output dtype inside:", y_half.dtype)
else:
    print("CPU session — AMP demo skipped (run this cell on a Colab GPU).")

### Weapon 4 — quantization: int8 weights for CPU serving

`quantize_dynamic` converts `Linear` weights to int8 at load time — smaller files, faster CPU
matmuls, tiny accuracy cost.

In [ ]:
import os
net_cpu = net.cpu().eval()
try:
    q8 = torch.quantization.quantize_dynamic(net_cpu, {nn.Linear}, dtype=torch.qint8)

    def size_mb(m):
        torch.save(m.state_dict(), "_tmp.pt"); s = os.path.getsize("_tmp.pt") / 1e6; os.remove("_tmp.pt"); return s

    xc = x.cpu()
    print(f"size:  fp32 {size_mb(net_cpu):.2f} MB  ->  int8 {size_mb(q8):.2f} MB")
    print(f"speed: fp32 {stopwatch(lambda: net_cpu(xc)):.2f} ms  ->  int8 {stopwatch(lambda: q8(xc)):.2f} ms")
    print(f"max output drift: {(net_cpu(xc) - q8(xc)).abs().max().item():.4f}")
except Exception as e:
    print(f"quantization unavailable ({type(e).__name__})")

### Weapon 5 — shipping: TorchScript

`torch.jit.trace` freezes the model into a self-contained artifact loadable from C++, mobile,
or a Python process with no model class definition — the standard handoff to production.

In [ ]:
os.makedirs("checkpoints", exist_ok=True)
example = torch.randn(1, 512)
scripted = torch.jit.trace(net_cpu, example)
scripted.save("checkpoints/speednet.pt")

revived = torch.jit.load("checkpoints/speednet.pt")
print("round-trip output identical:", torch.allclose(net_cpu(example), revived(example)))

class Predictor:
    """What your web service would import — no model class needed."""
    def __init__(self, path):
        self.m = torch.jit.load(path); self.m.eval()
    @torch.no_grad()
    def __call__(self, feats):
        return self.m(torch.as_tensor(feats, dtype=torch.float32).reshape(1, -1))

p = Predictor("checkpoints/speednet.pt")
print("served prediction shape:", p([0.0] * 512).shape)

In [ ]:
# ── this notebook's exercises (run once) ───────────────────────────────
_register("warmup", 5,
    lambda s: "sync" in s.lower() or "async" in s.lower(),
    "why must you call torch.cuda.synchronize() before timing GPU code? (mention sync/async)")
_register("quant_shrink", 15,
    lambda sizes: len(sizes) == 2 and sizes[1] < 0.5 * sizes[0],
    "quantize any Linear-heavy model; submit (fp32_mb, int8_mb) — int8 should be < half the size")
_register("script_match", 15,
    lambda ok: ok is True,
    "trace any model, save, reload, and submit torch.allclose(original(x), reloaded(x))")
_register("first_move", 10,
    lambda s: "measure" in s.lower() or "profile" in s.lower() or "benchmark" in s.lower() or "time" in s.lower(),
    "one word: what do you ALWAYS do before optimizing?")

In [ ]:
check("first_move", "measure")

## 🏆 Boss Challenges

Earn your XP — write each answer in a cell below and grade it with `check(...)`. When you're done, run `xp_report()`.

- `warmup` (5 XP) — why synchronize before timing GPU code?
- `quant_shrink` (15 XP) — quantize a model; submit `(fp32_mb, int8_mb)`.
- `script_match` (15 XP) — TorchScript round-trip; submit the `allclose` bool.

In [ ]:
# ⚔️ your attempts here...

# xp_report()